# JetBot Tape Line Follower (v2.4 — Decoupled Control Loop)

**New in this version:**
- **Decoupled control:** vision runs at camera rate (~10–30Hz), PID + motor commands run on a separate ~200Hz thread driven by the IMU. The derivative term comes from the gyro (fast), the proportional term from the camera (slow).
- **Mask preview toggle:** hide the mask view to skip JPEG-encoding the second image every frame.
- **Tuned defaults** baked into sliders (Blur=15, Canny=85/85, Dilate=15, ROI Height=0.45, Horizon Clip=0.25, Scan Rows=4).
- **Renamed ROI sliders** for clarity: `ROI Bot %` → `ROI Height`, `ROI Top %` → `Horizon Clip`.
- **Anti-windup integral with time-based decay** — works correctly at any control rate.
- **Loop-rate readout** so you can see what you're actually achieving.

**Color legend:** Blue = IMU, Red = Recording, Green active state.


## Slider reference

### Detection — Canny Edge mode
- **Blur K** — Gaussian blur kernel size before edge detection. Larger = smoother input, fewer spurious edges, but loses fine detail. *Must be odd.* Default: 15.
- **Canny Low** / **Canny High** — Hysteresis thresholds. Pixels with gradient ≥ High are seeded as edges; pixels with gradient ≥ Low are kept only if connected to a seeded edge. **High should be > Low.** Default: 85 / 85 (tuned by user; they're equal which collapses hysteresis to a single threshold — fine for your tape).
- **Dilate K** — Size of the morphological dilation kernel applied after Canny. Larger = thicker edges, more likely to merge nearby edges into a single tape blob. Run 2× per frame. Default: 15.

### Detection — Adaptive Threshold mode
- **Block Size** — Size of the local neighborhood used to compute the per-pixel threshold. Must be odd, ≥ 3. Larger = smoother regions, more robust to even lighting variation. Default: 11.
- **C Offset** — Constant subtracted from the local mean. Positive = stricter (fewer pixels classed as bright). Default: 2.
- **Invert** — Flip black↔white. Use if your tape is dark on a light surface vs. light on a dark surface.

### Shared
- **Min Run W** — Minimum width (in pixels) for a contiguous run of mask pixels to be considered part of a tape strip. Filters out tiny noise specks. Default: 3.

### ROI (Region of Interest)
The robot only analyzes a horizontal band of the camera frame. Two sliders define it:
- **ROI Height** — How tall the ROI band is, as a fraction of the full frame, anchored to the bottom. 0.45 means the bottom 45% of the frame is the candidate region. Default: 0.45.
- **Horizon Clip** — How much of the *top* of that band to trim off. 0.25 means trim the upper 25% of the frame (the part most distorted by perspective and most likely to contain horizon clutter). Default: 0.25.
- The actual analyzed band is the intersection: `[h·(1 − Height) : h·(1 − HorizonClip)]`.
- Visual: red line = upper ROI edge, orange line = lower edge of the horizon clip, gray vertical line = frame center.

### Lane scanning
- **Scan Rows** — Number of horizontal rows scanned across the ROI to find tape edges. More rows = smoother lane center, more CPU. Default: 4.
- **Near Wt** — Weight applied to the nearest scanned rows when computing `near_err`. Higher = robot pays more attention to what's right in front of it. Default: 2.0.

### PID / Steering
- **Kp** — Proportional gain. How aggressively to steer in proportion to current lane error. Too high = oscillates, too low = sluggish. Default: 0.4.
- **Ki** — Integral gain. Corrects steady-state offset (e.g., if the bot consistently drifts left, Ki nudges it right). Easy to over-tune. Default: 0.0.
- **Kd** — Derivative gain. Damps oscillation by penalizing the *rate of change* of error. With IMU enabled, this term uses gyro Z instead of visual derivative. Default: 0.1.

### Speed / Drive
- **Base Speed** — Forward speed when going straight (0–1 motor scale). Default: 0.25.
- **Curve Slow** — How much to reduce speed when steering. `speed = base_speed · (1 − curve_slow · |steer|)`. Default: 0.5.
- **Look-ahead** — Blend factor between near-error (what's right in front) and far-error (what's coming up). 0 = pure near, 1 = pure far. Higher = better turn-in but laggier on tight corners. Default: 0.3.
- **Min Wheel** — Floor applied to motor commands. Useful if your motors have a deadband (won't move below ~0.1). Negative values let the bot reverse a wheel for tighter turns. Default: 0.0.

### IMU (blue panel)
- **Gyro Blend** — Mix of visual derivative vs gyro derivative in the PID D term. 0 = pure vision (legacy), 1 = pure gyro. Default: 0.4.
- **Gyro Sign** — Flip the sign convention if turning right gives negative gyro Z. Default: +1.

### Recording (red panel)
- **● REC** — Toggle CSV logging. Files saved as `run_YYYYMMDD_HHMMSS.csv` in the working directory. Logs vision, IMU, control, and learner state per camera frame.

### Learner
- **Enable Learn** — Activate the bandit learner that adds small bias corrections on top of the PID.
- **ε (explore)** — Probability of taking a random action. Higher = explores more, lower = exploits learned policy.
- **α (lr)** — Q-table learning rate. Higher = adapts faster but noisier.
- **Spd Bonus** — Reward shaping weight on forward speed. Discourages the trivial "go slow to stay centered" solution.


In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import json
import time
import csv
import os
import math
import threading
from datetime import datetime
from jetbot import Robot, Camera

try:
    from smbus2 import SMBus
    _SMBUS_AVAILABLE = True
except ImportError:
    _SMBUS_AVAILABLE = False
    print('[WARN] smbus2 not installed. IMU will be disabled. '
          'Run:  pip install smbus2')

In [2]:
camera = Camera.instance(width=300, height=300, fps=10)
robot = Robot()

In [3]:
# ── MPU-6050 IMU driver ──
MPU6050_ADDR = 0x68
I2C_BUS      = 0
PWR_MGMT_1   = 0x6B
GYRO_CONFIG  = 0x1B
GYRO_XOUT_H  = 0x43
ACCEL_XOUT_H = 0x3B
GYRO_SCALE_250 = 131.0       # LSB / (deg/s) at ±250 dps

class IMU:
    def __init__(self, bus_num=I2C_BUS, addr=MPU6050_ADDR):
        self.addr = addr
        self.ok = False
        self.bus = None
        self.gyro_z_bias = 0.0
        self._lock = threading.Lock()  # I2C bus is shared between threads
        if not _SMBUS_AVAILABLE:
            return
        try:
            self.bus = SMBus(bus_num)
            self.bus.write_byte_data(self.addr, PWR_MGMT_1, 0x00)
            time.sleep(0.05)
            self.bus.write_byte_data(self.addr, GYRO_CONFIG, 0x00)
            self.ok = True
            print(f'[IMU] connected at 0x{self.addr:02X} on bus {bus_num}')
        except Exception as e:
            print(f'[IMU] init failed: {e}')

    def _read_word_signed(self, reg):
        hi = self.bus.read_byte_data(self.addr, reg)
        lo = self.bus.read_byte_data(self.addr, reg + 1)
        v = (hi << 8) | lo
        return v - 0x10000 if v >= 0x8000 else v

    def read_gyro_z(self):
        if not self.ok:
            return 0.0
        with self._lock:
            try:
                raw = self._read_word_signed(GYRO_XOUT_H + 4)
                return (raw / GYRO_SCALE_250) - self.gyro_z_bias
            except Exception:
                return 0.0

    def read_accel_xy(self):
        if not self.ok:
            return 0, 0
        with self._lock:
            try:
                ax = self._read_word_signed(ACCEL_XOUT_H)
                ay = self._read_word_signed(ACCEL_XOUT_H + 2)
                return ax, ay
            except Exception:
                return 0, 0

    def calibrate(self, samples=200):
        if not self.ok:
            return
        readings = []
        for _ in range(samples):
            with self._lock:
                try:
                    raw = self._read_word_signed(GYRO_XOUT_H + 4)
                    readings.append(raw / GYRO_SCALE_250)
                except Exception:
                    pass
            time.sleep(0.005)
        if readings:
            self.gyro_z_bias = float(np.mean(readings))
            print(f'[IMU] gyro Z bias = {self.gyro_z_bias:+.3f} dps '
                  f'({len(readings)} samples)')

imu = IMU()
if imu.ok:
    print('[IMU] keep the bot still — calibrating...')
    imu.calibrate()

[IMU] connected at 0x68 on bus 0
[IMU] keep the bot still — calibrating...
[IMU] gyro Z bias = -0.763 dps (200 samples)


In [4]:
LEARNER_CFG = "bandit_qtable.json"
ERR_EDGES   = np.array([-0.5, -0.15, 0.15, 0.5])
DERR_EDGES  = np.array([-0.08, 0.08])
GYRO_EDGES  = np.array([-30.0, 30.0])
STEER_DELTAS = np.array([-0.10, 0.0, +0.10])
SPEED_DELTAS = np.array([-0.05, 0.0, +0.05])
N_ERR, N_DERR, N_GYRO = len(ERR_EDGES)+1, len(DERR_EDGES)+1, len(GYRO_EDGES)+1
N_STEER, N_SPEED = len(STEER_DELTAS), len(SPEED_DELTAS)

def bin_idx(value, edges):
    return int(np.searchsorted(edges, value))

class BanditLearner:
    def __init__(self):
        self.Q = np.zeros((N_ERR, N_DERR, N_GYRO, N_STEER, N_SPEED), dtype=np.float32)
        self.N = np.zeros_like(self.Q, dtype=np.int32)
        self.alpha = 0.15; self.epsilon = 0.20; self.speed_bonus = 0.3
        self.last_state = None; self.last_action = None

    def state_from(self, error, derror, gyro_z):
        return (bin_idx(error, ERR_EDGES), bin_idx(derror, DERR_EDGES),
                bin_idx(gyro_z, GYRO_EDGES))

    def select_action(self, state):
        ei, di, gi = state
        if np.random.rand() < self.epsilon:
            si, spi = np.random.randint(N_STEER), np.random.randint(N_SPEED)
        else:
            slab = self.Q[ei, di, gi]
            si, spi = np.unravel_index(np.argmax(slab), slab.shape)
        self.last_state = state; self.last_action = (int(si), int(spi))
        return STEER_DELTAS[si], SPEED_DELTAS[spi]

    def observe_reward(self, reward):
        if self.last_state is None: return
        ei, di, gi = self.last_state; si, spi = self.last_action
        old = self.Q[ei, di, gi, si, spi]
        self.Q[ei, di, gi, si, spi] = old + self.alpha * (reward - old)
        self.N[ei, di, gi, si, spi] += 1

    def reset_episode(self):
        self.last_state = None; self.last_action = None

    def save(self, path=LEARNER_CFG):
        with open(path, "w") as f:
            json.dump({"Q": self.Q.tolist(), "N": self.N.tolist(),
                       "alpha": self.alpha, "epsilon": self.epsilon,
                       "speed_bonus": self.speed_bonus}, f)

    def load(self, path=LEARNER_CFG):
        if not os.path.exists(path): return False
        with open(path) as f: payload = json.load(f)
        try:
            Q = np.array(payload["Q"], dtype=np.float32)
            N = np.array(payload["N"], dtype=np.int32)
            if Q.shape == self.Q.shape: self.Q = Q; self.N = N
            else: print(f'[Learner] shape mismatch, starting fresh.')
        except Exception as e: print(f'[Learner] load failed: {e}'); return False
        self.alpha = payload.get("alpha", self.alpha)
        self.epsilon = payload.get("epsilon", self.epsilon)
        self.speed_bonus = payload.get("speed_bonus", self.speed_bonus)
        return True

learner = BanditLearner()
learner.load()

False

In [5]:
REC_FIELDS = [
    "t_sec", "frame_idx",
    "imu_enabled", "drive_enabled", "learn_enabled",
    "error", "deriv", "lane_center_x", "rows_seen", "line_lost",
    "gyro_z", "accel_x", "accel_y",
    "steer", "speed", "lm", "rm",
    "learner_state", "learner_action", "reward",
]

class Recorder:
    def __init__(self, flush_every=30):
        self.active = False; self.buf = []; self.flush_every = flush_every
        self.path = None; self.fh = None; self.writer = None
        self.t0 = None; self.frame_idx = 0
        self._lock = threading.Lock()

    def start(self):
        if self.active: return
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.path = f"run_{ts}.csv"
        self.fh = open(self.path, "w", newline="")
        self.writer = csv.DictWriter(self.fh, fieldnames=REC_FIELDS)
        self.writer.writeheader()
        self.t0 = time.monotonic(); self.frame_idx = 0
        self.active = True; self.buf = []

    def stop(self):
        if not self.active: return
        with self._lock:
            self._flush_locked()
        try: self.fh.close()
        except Exception: pass
        self.active = False; self.fh = None; self.writer = None

    def _flush_locked(self):
        if self.buf and self.writer:
            self.writer.writerows(self.buf); self.fh.flush(); self.buf = []

    def log(self, row):
        if not self.active: return
        with self._lock:
            self.frame_idx += 1
            row["t_sec"] = round(time.monotonic() - self.t0, 4)
            row["frame_idx"] = self.frame_idx
            self.buf.append(row)
            if len(self.buf) >= self.flush_every:
                self._flush_locked()

recorder = Recorder()

In [6]:
# ── Shared state between camera (slow) and control (fast) threads ──
# Camera thread WRITES vision_state. Control thread READS it.
# Control thread WRITES motor commands. Camera thread reads them only for display.

class SharedState:
    """Minimal lock-protected container for cross-thread state."""
    def __init__(self):
        self.lock = threading.Lock()
        # Vision (written by camera thread)
        self.error = 0.0
        self.lane_valid = False
        self.lane_center_x = 0
        self.rows_seen = 0
        self.vision_t = 0.0      # timestamp of last vision update
        # Control (written by fast thread, read by display)
        self.steer = 0.0
        self.speed = 0.0
        self.lm = 0.0
        self.rm = 0.0
        self.gyro_z = 0.0
        self.actual_hz = 0.0     # measured fast-loop rate

    def set_vision(self, error, valid, lane_x, rows):
        with self.lock:
            self.error = error
            self.lane_valid = valid
            self.lane_center_x = lane_x
            self.rows_seen = rows
            self.vision_t = time.monotonic()

    def get_vision(self):
        with self.lock:
            return self.error, self.lane_valid, self.vision_t

    def set_control(self, steer, speed, lm, rm, gyro_z, hz):
        with self.lock:
            self.steer, self.speed, self.lm, self.rm = steer, speed, lm, rm
            self.gyro_z, self.actual_hz = gyro_z, hz

    def get_control_snapshot(self):
        with self.lock:
            return dict(steer=self.steer, speed=self.speed,
                        lm=self.lm, rm=self.rm,
                        gyro_z=self.gyro_z, hz=self.actual_hz,
                        error=self.error, lane_valid=self.lane_valid,
                        lane_center_x=self.lane_center_x, rows_seen=self.rows_seen)

state = SharedState()

class ControlLoop:
    """Fast thread: reads IMU + latest vision error, runs PID, drives motors.
    Target rate is set by `target_hz`. Actual rate is measured and exposed."""
    def __init__(self, target_hz=200.0):
        self.target_hz = target_hz
        self.thread = None
        self.stop_event = threading.Event()
        self.integral = 0.0
        self.prev_error = 0.0
        self.tau_i = 1.0   # integral leak time constant in seconds

    def start(self):
        if self.thread is not None and self.thread.is_alive():
            return
        self.stop_event.clear()
        self.integral = 0.0
        self.prev_error = 0.0
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def stop(self, timeout=1.0):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=timeout)
        self.thread = None
        try:
            robot.stop()
        except Exception:
            pass

    def _run(self):
        period = 1.0 / self.target_hz
        last_t = time.monotonic()
        # EMA of measured loop rate
        hz_ema = self.target_hz
        VISION_STALE_S = 0.5   # if no vision for >500ms, assume LINE LOST

        while not self.stop_event.is_set():
            t0 = time.monotonic()
            dt = max(1e-4, t0 - last_t)
            last_t = t0

            # --- Read latest vision (cheap, just a lock) ---
            err, valid, vt = state.get_vision()
            vision_age = t0 - vt
            if vision_age > VISION_STALE_S:
                valid = False

            # --- Read IMU ---
            if imu_enabled.value and imu.ok:
                gyro_z = imu.read_gyro_z() * gyro_z_sign.value
            else:
                gyro_z = 0.0

            if not valid:
                # No tape — coast to stop, reset integral
                self.integral = 0.0
                self.prev_error = 0.0
                if drive_enabled.value:
                    robot.stop()
                state.set_control(0.0, 0.0, 0.0, 0.0, gyro_z, hz_ema)
                # sleep & continue
                slept = period - (time.monotonic() - t0)
                if slept > 0: time.sleep(slept)
                continue

            # --- PID ---
            # Time-based integral leak: I ← I·exp(-dt/τ) + err·dt
            decay = math.exp(-dt / self.tau_i)
            self.integral = self.integral * decay + err * dt
            # Anti-windup clamp
            self.integral = max(-2.0, min(2.0, self.integral))

            # Derivative: blend visual deriv (per-vision-frame) and gyro deriv
            # (per fast-loop tick). Gyro dominates because it's fresh.
            visual_deriv = (err - self.prev_error)  # per fast tick
            self.prev_error = err
            gyro_deriv = gyro_z / 180.0
            gb = gyro_blend.value if (imu_enabled.value and imu.ok) else 0.0
            deriv = (1.0 - gb) * visual_deriv + gb * gyro_deriv

            steer = (kp.value * err
                     + ki.value * self.integral
                     + kd.value * deriv)
            steer = max(-1.0, min(1.0, steer))

            speed = base_speed.value * (1.0 - curve_slow.value * abs(steer))

            lm = speed + steer
            rm = speed - steer
            mw = min_wheel.value
            lm = max(mw, min(1.0, lm))
            rm = max(mw, min(1.0, rm))

            if drive_enabled.value:
                try:
                    robot.set_motors(lm, rm)
                except Exception:
                    pass

            # Update measured loop rate (EMA)
            measured_hz = 1.0 / dt
            hz_ema = 0.95 * hz_ema + 0.05 * measured_hz
            state.set_control(steer, speed, lm, rm, gyro_z, hz_ema)

            # Sleep to target rate
            slept = period - (time.monotonic() - t0)
            if slept > 0:
                time.sleep(slept)

control_loop = ControlLoop(target_hz=200.0)

In [7]:
# Camera feeds
cam_img  = widgets.Image(format='jpeg', width=300, height=300)
mask_img = widgets.Image(format='jpeg', width=300, height=300)
info_lbl = widgets.HTML(value='<i>Waiting for frames...</i>')

# Mode dropdown
mode = widgets.Dropdown(options=['Canny Edge', 'Adaptive Threshold'],
                        value='Canny Edge', description='Mode')

# Canny sliders (TUNED DEFAULTS)
blur_k     = widgets.IntSlider(value=15,  min=1, max=21, step=2, description='Blur K')
canny_low  = widgets.IntSlider(value=85,  min=0, max=255, description='Canny Low')
canny_high = widgets.IntSlider(value=85,  min=0, max=255, description='Canny High')
dilate_k   = widgets.IntSlider(value=15,  min=1, max=15, step=2, description='Dilate K')

# Adaptive threshold sliders
block_size = widgets.IntSlider(value=11,  min=3,  max=51, step=2, description='Block Size')
c_offset   = widgets.IntSlider(value=2,   min=-20, max=20, description='C Offset')
invert     = widgets.ToggleButton(value=False, description='Invert',
                button_style='', layout=widgets.Layout(width='100px'))

# Shared sliders (TUNED DEFAULTS, RENAMED ROI)
min_run_w     = widgets.IntSlider(value=3,    min=1, max=30, description='Min Run W')
roi_pct       = widgets.FloatSlider(value=0.45, min=0.1, max=1.0, step=0.05,
                                    description='ROI Height')
roi_top_pct   = widgets.FloatSlider(value=0.25, min=0.0, max=0.6, step=0.05,
                                    description='Horizon Clip')
scan_rows     = widgets.IntSlider(value=4,    min=3, max=30, description='Scan Rows')
near_weight   = widgets.FloatSlider(value=2.0,  min=1.0, max=6.0, step=0.1, description='Near Wt')
kp            = widgets.FloatSlider(value=0.4,  min=0.0, max=2.0, step=0.01, description='Kp')
ki            = widgets.FloatSlider(value=0.0,  min=0.0, max=0.5, step=0.005, description='Ki')
kd            = widgets.FloatSlider(value=0.1,  min=0.0, max=1.0, step=0.01, description='Kd')
base_speed    = widgets.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01, description='Base Speed')
curve_slow    = widgets.FloatSlider(value=0.5,  min=0.0, max=1.0, step=0.05, description='Curve Slow')
lookahead     = widgets.FloatSlider(value=0.3,  min=0.0, max=1.0, step=0.05, description='Look-ahead')
min_wheel     = widgets.FloatSlider(value=0.0,  min=-0.2, max=0.15, step=0.01, description='Min Wheel')

# Drive toggle — ALSO STARTS/STOPS THE FAST CONTROL THREAD
drive_enabled = widgets.ToggleButton(
    value=False, description='Enable Drive',
    button_style='danger', icon='car',
    layout=widgets.Layout(width='160px', height='40px'))

def on_drive_toggle(change):
    if change['new']:
        drive_enabled.button_style = 'success'
        drive_enabled.description = 'Driving!'
        control_loop.start()
    else:
        drive_enabled.button_style = 'danger'
        drive_enabled.description = 'Enable Drive'
        control_loop.stop()
drive_enabled.observe(on_drive_toggle, names='value')

# Show/hide mode-specific
canny_box    = widgets.VBox([blur_k, canny_low, canny_high, dilate_k])
adaptive_box = widgets.VBox([block_size, c_offset, invert])
adaptive_box.layout.display = 'none'
def on_mode_change(change):
    canny_box.layout.display    = '' if change['new'] == 'Canny Edge' else 'none'
    adaptive_box.layout.display = 'none' if change['new'] == 'Canny Edge' else ''
mode.observe(on_mode_change, names='value')

# ── Mask preview toggle (NEW) ──
show_mask = widgets.ToggleButton(
    value=True, description='Show Mask',
    button_style='info', icon='eye',
    layout=widgets.Layout(width='140px'))

def on_show_mask_toggle(change):
    if change['new']:
        show_mask.description = 'Show Mask'
        show_mask.button_style = 'info'
        mask_img.layout.display = ''
    else:
        show_mask.description = 'Hide Mask'
        show_mask.button_style = ''
        mask_img.layout.display = 'none'  # hide widget AND skip encode
show_mask.observe(on_show_mask_toggle, names='value')

# ── IMU panel (BLUE) ──
imu_enabled = widgets.ToggleButton(value=False, description='Enable IMU',
    button_style='', icon='compass',
    layout=widgets.Layout(width='160px', height='40px'),
    disabled=(not imu.ok))
def on_imu_toggle(change):
    if change['new']:
        imu_enabled.button_style = 'success'; imu_enabled.description = 'IMU ON'
    else:
        imu_enabled.button_style = ''; imu_enabled.description = 'Enable IMU'
imu_enabled.observe(on_imu_toggle, names='value')
gyro_blend  = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='Gyro Blend')
gyro_z_sign = widgets.Dropdown(options=[('+1', 1.0), ('-1', -1.0)], value=1.0, description='Gyro Sign')
recal_btn   = widgets.Button(description='Recalibrate', button_style='info', icon='balance-scale')
imu_status  = widgets.HTML(value=(
    '<span style="color:#0a7">Connected at 0x68</span>' if imu.ok
    else '<span style="color:#a00">Not connected</span>'))
imu_live    = widgets.HTML(value='<i>idle</i>')
def _recalibrate(_):
    if imu.ok:
        imu_status.value = '<span style="color:#a70">Calibrating — keep still...</span>'
        imu.calibrate()
        imu_status.value = (f'<span style="color:#0a7">Recalibrated, '
                            f'bias={imu.gyro_z_bias:+.3f} dps</span>')
recal_btn.on_click(_recalibrate)
imu_box = widgets.VBox([
    widgets.HTML('<div style="background:#e8f0fe;padding:6px 8px;border-radius:4px;'
                 'border-left:4px solid #2962ff"><b style="color:#1a3a8a">'
                 '🧭 IMU (MPU-6050)</b></div>'),
    imu_enabled, gyro_blend, gyro_z_sign, recal_btn, imu_status, imu_live,
])

# ── Recording panel (RED) ──
rec_btn = widgets.ToggleButton(value=False, description='● REC',
    button_style='', icon='circle',
    layout=widgets.Layout(width='160px', height='40px'))
rec_status = widgets.HTML(value='<i>not recording</i>')
def on_rec_toggle(change):
    if change['new']:
        recorder.start()
        rec_btn.button_style = 'danger'; rec_btn.description = '■ STOP REC'
        rec_status.value = (f'<span style="color:#c00"><b>● RECORDING</b></span><br>'
                            f'<small>file: {recorder.path}</small>')
    else:
        recorder.stop()
        rec_btn.button_style = ''; rec_btn.description = '● REC'
        rec_status.value = (f'<i>stopped — saved {recorder.frame_idx} frames'
                            f' to {recorder.path}</i>' if recorder.path
                            else '<i>not recording</i>')
rec_btn.observe(on_rec_toggle, names='value')
rec_box = widgets.VBox([
    widgets.HTML('<div style="background:#fde8e8;padding:6px 8px;border-radius:4px;'
                 'border-left:4px solid #c00"><b style="color:#8a1a1a">'
                 '⏺ Recording</b></div>'),
    rec_btn, rec_status,
])

# ── Loop rate display (new) ──
loop_hz_lbl = widgets.HTML(value='<small>control loop: idle</small>')

# ── Learner panel ──
learn_enabled = widgets.ToggleButton(value=False, description='Enable Learn',
    button_style='', icon='brain', layout=widgets.Layout(width='160px', height='40px'))
epsilon_slider     = widgets.FloatSlider(value=learner.epsilon, min=0.0, max=0.5, step=0.01, description='ε (explore)')
alpha_slider       = widgets.FloatSlider(value=learner.alpha,   min=0.01, max=0.5, step=0.01, description='α (lr)')
speed_bonus_slider = widgets.FloatSlider(value=learner.speed_bonus, min=0.0, max=1.0, step=0.05, description='Spd Bonus')
def _push_hyperparams(_=None):
    learner.epsilon = epsilon_slider.value
    learner.alpha = alpha_slider.value
    learner.speed_bonus = speed_bonus_slider.value
for w in (epsilon_slider, alpha_slider, speed_bonus_slider):
    w.observe(_push_hyperparams, names='value')
save_q_btn = widgets.Button(description='Save Q', button_style='info', icon='save')
load_q_btn = widgets.Button(description='Load Q', button_style='warning', icon='folder-open')
reset_q_btn = widgets.Button(description='Reset Q', button_style='danger', icon='trash')
q_status = widgets.Label(value=('Q loaded' if learner.N.sum() else 'Q empty'))
def _save_q(_): learner.save(); q_status.value = f'Saved (visits={int(learner.N.sum())})'
def _load_q(_):
    ok = learner.load()
    q_status.value = (f'Loaded (visits={int(learner.N.sum())})' if ok else 'No file')
def _reset_q(_):
    learner.Q[:] = 0; learner.N[:] = 0; learner.reset_episode(); q_status.value = 'Q reset'
save_q_btn.on_click(_save_q); load_q_btn.on_click(_load_q); reset_q_btn.on_click(_reset_q)
learner_info = widgets.HTML(value='<i>Learner idle</i>')
learner_box = widgets.VBox([
    widgets.HTML('<hr style="margin:4px 0"><b>Bandit Learner</b>'),
    learn_enabled, epsilon_slider, alpha_slider, speed_bonus_slider,
    widgets.HBox([save_q_btn, load_q_btn, reset_q_btn]),
    q_status, learner_info,
])

# ── Save / Load PID config ──
save_btn = widgets.Button(description='Save', button_style='info', icon='save')
load_btn = widgets.Button(description='Load', button_style='warning', icon='folder-open')
status_lbl = widgets.Label(value='')
CFG = 'line_follower_config.json'
ALL_PARAMS = [('mode',mode),('blur_k',blur_k),('canny_low',canny_low),
              ('canny_high',canny_high),('dilate_k',dilate_k),
              ('block_size',block_size),('c_offset',c_offset),('invert',invert),
              ('min_run_w',min_run_w),('roi_pct',roi_pct),('roi_top_pct',roi_top_pct),
              ('scan_rows',scan_rows),('near_weight',near_weight),
              ('kp',kp),('ki',ki),('kd',kd),('base_speed',base_speed),
              ('curve_slow',curve_slow),('lookahead',lookahead),('min_wheel',min_wheel),
              ('gyro_blend',gyro_blend),('gyro_z_sign',gyro_z_sign),
              ('show_mask', show_mask)]
def save_config(_):
    with open(CFG,'w') as f:
        json.dump({k: w.value for k,w in ALL_PARAMS}, f, indent=2)
    status_lbl.value = 'Saved!'
def load_config(_):
    try:
        with open(CFG) as f: cfg = json.load(f)
        for k, w in ALL_PARAMS:
            if k in cfg:
                try: w.value = cfg[k]
                except Exception: pass
        status_lbl.value = 'Loaded!'
    except FileNotFoundError:
        status_lbl.value = 'No config file.'
save_btn.on_click(save_config); load_btn.on_click(load_config)

# Layout
left_col = widgets.VBox([
    widgets.HBox([cam_img, mask_img]),
    widgets.HBox([show_mask, loop_hz_lbl]),
    info_lbl,
    widgets.HBox([imu_box, rec_box], layout=widgets.Layout(gap='12px')),
])
right_col = widgets.VBox([
    mode, widgets.HTML('<b>Detection</b>'),
    canny_box, adaptive_box, min_run_w,
    widgets.HTML('<hr style="margin:4px 0"><b>PID / Steering</b>'),
    roi_pct, roi_top_pct, scan_rows, near_weight, kp, ki, kd,
    widgets.HTML('<hr style="margin:4px 0"><b>Speed / Drive</b>'),
    base_speed, curve_slow, lookahead, min_wheel,
    drive_enabled,
    widgets.HBox([save_btn, load_btn, status_lbl]),
    learner_box,
], layout=widgets.Layout(padding='0 0 0 12px'))

display(widgets.HBox([left_col, right_col]))

# Cached dilate kernel (small perf win)
_dilate_kernels = {}
def _get_dilate_kernel(k):
    k = k | 1
    if k not in _dilate_kernels:
        _dilate_kernels[k] = np.ones((k, k), np.uint8)
    return _dilate_kernels[k]

_recent_rewards = []

def get_runs(row):
    padded = np.concatenate(([0], row, [0]))
    diff = np.diff(padded.astype(np.int16))
    starts = np.where(diff > 0)[0]
    ends   = np.where(diff < 0)[0]
    return list(zip(starts, ends))

def find_lane_center(mask):
    rh, rw = mask.shape
    n = scan_rows.value
    row_indices = np.linspace(rh - 1, 0, n, dtype=int)
    mrw = min_run_w.value
    points = []
    for r in row_indices:
        runs = get_runs(mask[r])
        runs = [(s, e) for s, e in runs if (e - s) >= mrw]
        if len(runs) >= 2:
            li = int(runs[0][1] - 1); ri = int(runs[-1][0])
            cx = (li + ri) // 2
            points.append((int(r), li, ri, cx))
        elif len(runs) == 1:
            s, e = runs[0]
            lx = int(s); rx = int(e - 1); cx = (lx + rx) // 2
            points.append((int(r), lx, rx, cx))
    return points

def process_frame(change):
    """CAMERA THREAD ONLY. Computes vision, writes to shared state.
    Does NOT run PID or set motors — that's the control loop's job."""
    global _recent_rewards
    frame = change['new']
    h, w = frame.shape[:2]

    roi_top    = int(h * (1.0 - roi_pct.value))
    roi_bottom = int(h * (1.0 - roi_top_pct.value))
    if roi_bottom <= roi_top + 10:
        roi_bottom = roi_top + 10
    roi = frame[roi_top:roi_bottom, :]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    bk = blur_k.value | 1
    blurred = cv2.GaussianBlur(gray, (bk, bk), 0)

    if mode.value == 'Canny Edge':
        edges = cv2.Canny(blurred, canny_low.value, canny_high.value)
        mask = cv2.dilate(edges, _get_dilate_kernel(dilate_k.value), iterations=2)
    else:
        bs = block_size.value | 1
        if bs < 3: bs = 3
        thresh_type = cv2.THRESH_BINARY_INV if not invert.value else cv2.THRESH_BINARY
        mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     thresh_type, bs, c_offset.value)
        kern = _get_dilate_kernel(3)
        mask = cv2.erode(mask, kern, iterations=1)
        mask = cv2.dilate(mask, kern, iterations=2)

    points = find_lane_center(mask)

    ov = frame.copy()
    cv2.line(ov, (0, roi_top), (w, roi_top), (0, 0, 255), 2)
    cv2.line(ov, (0, roi_bottom), (w, roi_bottom), (0, 165, 255), 2)
    cv2.line(ov, (w // 2, roi_top), (w // 2, roi_bottom), (128, 128, 128), 1)

    log_row = {f: None for f in REC_FIELDS}
    log_row.update({
        "imu_enabled": int(imu_enabled.value and imu.ok),
        "drive_enabled": int(drive_enabled.value),
        "learn_enabled": int(learn_enabled.value),
        "line_lost": 0,
    })

    if len(points) >= 2:
        for r, lx, rx, cx in points:
            yr = r + roi_top
            cv2.circle(ov, (lx, yr), 4, (0, 0, 255), -1)
            cv2.circle(ov, (rx, yr), 4, (255, 0, 0), -1)
            cv2.circle(ov, (cx, yr), 5, (0, 255, 0), -1)
        for i in range(len(points) - 1):
            y1 = points[i][0] + roi_top; y2 = points[i+1][0] + roi_top
            cv2.line(ov, (points[i][3], y1), (points[i+1][3], y2), (0, 255, 0), 2)

        mid = len(points) // 2
        near_pts = points[:mid] if mid > 0 else points
        far_pts  = points[mid:]
        nw = near_weight.value
        near_weights = np.linspace(nw, 1.0, len(near_pts))
        near_centers = np.array([p[3] for p in near_pts])
        near_cx = np.average(near_centers, weights=near_weights)
        near_err = (near_cx - w / 2) / (w / 2)
        far_centers = np.array([p[3] for p in far_pts])
        far_cx = np.mean(far_centers)
        far_err = (far_cx - w / 2) / (w / 2)
        la = lookahead.value
        error = (1.0 - la) * near_err + la * far_err
        blend_x = int((1.0 - la) * near_cx + la * far_cx)

        # ── Push vision state to control loop ──
        state.set_vision(error, True, blend_x, len(points))

        # ── Bandit learner: still runs on camera thread ──
        # Read current control state for reward shaping
        ctl = state.get_control_snapshot()
        if learn_enabled.value:
            reward = -abs(error) + learner.speed_bonus * ctl['speed']
            learner.observe_reward(reward)
            _recent_rewards.append(reward)
            if len(_recent_rewards) > 50: _recent_rewards.pop(0)
            # Note: learner state uses gyro from latest control snapshot
            st = learner.state_from(error, error - (learner.last_state[0]
                                                    if learner.last_state else 0)*0,
                                    ctl['gyro_z'])
            sb, vb = learner.select_action(st)
            avg_r = float(np.mean(_recent_rewards)) if _recent_rewards else 0.0
            learner_info.value = (
                f'<b>LEARN</b> state={st} bias=(s{sb:+.2f},v{vb:+.2f}) '
                f'r̄={avg_r:+.2f} visits={int(learner.N.sum())}')
            log_row["learner_state"]  = str(st)
            log_row["learner_action"] = f"({sb:+.2f},{vb:+.2f})"
            log_row["reward"]         = round(reward, 4)
        else:
            if _recent_rewards: _recent_rewards = []
            learner_info.value = '<i>Learner idle</i>'

        # Display markers
        bot_y = points[0][0] + roi_top
        top_y = far_pts[0][0] + roi_top if far_pts else bot_y
        cv2.circle(ov, (int(near_cx), bot_y), 10, (0, 255, 255), -1)
        cv2.circle(ov, (int(far_cx), top_y), 8, (255, 255, 0), 2)
        cv2.line(ov, (w // 2, bot_y), (blend_x, bot_y), (255, 0, 255), 2)

        # Read freshest control snapshot for the info label
        info_lbl.value = (f'<b style="color:green">TRACKING</b> &nbsp; '
                         f'err:{error:+.2f} &nbsp; steer:{ctl["steer"]:+.2f} &nbsp; '
                         f'spd:{ctl["speed"]:.2f} &nbsp; '
                         f'L:{ctl["lm"]:.2f} R:{ctl["rm"]:.2f} &nbsp; '
                         f'rows:{len(points)} &nbsp; gz:{ctl["gyro_z"]:+.0f}dps')
        loop_hz_lbl.value = f'<small>control loop: {ctl["hz"]:.0f} Hz</small>'

        # Fill log
        ax, ay = imu.read_accel_xy() if (imu_enabled.value and imu.ok) else (0, 0)
        log_row.update({
            "error": round(error, 4),
            "deriv": round(error - 0.0, 4),  # camera-thread deriv (limited use)
            "lane_center_x": blend_x,
            "rows_seen": len(points),
            "gyro_z": round(ctl["gyro_z"], 3),
            "accel_x": ax, "accel_y": ay,
            "steer": round(ctl["steer"], 4),
            "speed": round(ctl["speed"], 4),
            "lm": round(ctl["lm"], 4),
            "rm": round(ctl["rm"], 4),
        })
    else:
        # No tape — tell control loop
        state.set_vision(0.0, False, 0, 0)
        log_row["line_lost"] = 1
        learner.reset_episode()
        cv2.putText(ov, 'LINE LOST', (10, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        info_lbl.value = '<b style="color:red">LINE LOST</b> - motors stopped'
        ctl = state.get_control_snapshot()
        loop_hz_lbl.value = f'<small>control loop: {ctl["hz"]:.0f} Hz</small>'

    # IMU live readout (cheap)
    if imu_enabled.value and imu.ok:
        ctl = state.get_control_snapshot() if 'ctl' not in dir() else ctl
        imu_live.value = f'<small>gz={ctl["gyro_z"]:+6.1f} dps</small>'
    else:
        imu_live.value = '<i>idle</i>'

    recorder.log(log_row)

    _, j1 = cv2.imencode('.jpg', ov)
    cam_img.value = j1.tobytes()

    # ── Mask preview: SKIP encode if hidden ──
    if show_mask.value:
        mask_full = np.zeros((h, w), np.uint8)
        mask_full[roi_top:roi_bottom, :] = mask
        _, j2 = cv2.imencode('.jpg', cv2.cvtColor(mask_full, cv2.COLOR_GRAY2BGR))
        mask_img.value = j2.tobytes()

camera.observe(process_frame, names='value')
print('Dashboard active. Vision runs on camera thread; PID + motors on 200Hz control thread.')
print('Toggle Enable Drive to start the control loop.')

Dashboard active. Vision runs on camera thread; PID + motors on 200Hz control thread.
Toggle Enable Drive to start the control loop.


In [8]:
camera.unobserve_all()
control_loop.stop()
robot.stop()
drive_enabled.value = False
imu_enabled.value = False
if rec_btn.value:
    rec_btn.value = False
print('Stopped.')

Stopped.


In [ ]:
import glob
runs = sorted(glob.glob("run_*.csv"))
if not runs:
    print("No recordings yet.")
else:
    last = runs[-1]
    print(f"Most recent: {last}")
    with open(last) as f: rows = list(csv.DictReader(f))
    print(f"Frames: {len(rows)}")
    if rows:
        line_lost = sum(1 for r in rows if r["line_lost"] == "1")
        errs = [float(r["error"]) for r in rows if r["error"] not in (None, "")]
        gz   = [float(r["gyro_z"]) for r in rows if r["gyro_z"] not in (None, "")]
        print(f"  line lost frames: {line_lost} ({100*line_lost/len(rows):.1f}%)")
        if errs:
            print(f"  |error| mean: {np.mean(np.abs(errs)):.3f}, max: {max(np.abs(errs)):.3f}")
        if gz:
            print(f"  gyro Z range: {min(gz):+.1f} to {max(gz):+.1f} dps")